# 03 Model Training and Validation

Purpose: create the internal company-level validation split, select hyperparameters using validation PR-AUC, evaluate the seven source models, and save validation outputs. Inputs: `train.csv`. Outputs: validation comparison, classification reports, model specification, and validation predictions.

In [1]:

from __future__ import annotations

import ast
import json
import math
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, PercentFormatter

current_path = Path.cwd().resolve()
if current_path.name == 'notebooks':
    project_root = current_path.parent
else:
    project_root = current_path
assert project_root.name == 'ml-finance-bankruptcy-analysis', f'Unexpected project root: {project_root}'

DATA_DIR = project_root / 'data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
OUTPUTS_DIR = project_root / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
TABLES_DIR = OUTPUTS_DIR / 'tables'
REPORT_DIR = OUTPUTS_DIR / 'report'
PAPER_TABLES_DIR = OUTPUTS_DIR / 'paper' / 'tables'
PAPER_FIGURES_DIR = OUTPUTS_DIR / 'paper' / 'figures'
for path in [RAW_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR, TABLES_DIR, REPORT_DIR, PAPER_TABLES_DIR, PAPER_FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = RAW_DATA_DIR / 'american_bankruptcy.csv'
MODEL_DATASET_PATH = PROCESSED_DATA_DIR / 'model_dataset.csv'
TRAIN_DATA_PATH = PROCESSED_DATA_DIR / 'train.csv'
TEST_DATA_PATH = PROCESSED_DATA_DIR / 'test.csv'
COMPANY_COLUMN = 'company_name'
RAW_TARGET_COLUMN = 'status_label'
TARGET_COLUMN = 'failed'
YEAR_COLUMN = 'year'
ALIVE_LABEL = 'alive'
FAILED_LABEL = 'failed'
RANDOM_STATE = 42
OUTER_TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
LOGISTIC_C_GRID = [0.01, 0.1, 1.0, 10.0]
PCA_COMPONENT_GRID = [2, 3, 5, 8, 10, 12, 15, 18]
EXPECTED_MODEL_ORDER = [
    'Majority-class baseline',
    'Logistic Regression',
    'L1 Regularized Logistic Regression',
    'L2 Regularized Logistic Regression',
    'Decision Tree',
    'Random Forest',
    'Gradient Boosting',
]
PREDICTION_TABLE_COLUMNS = ['model', COMPANY_COLUMN, YEAR_COLUMN, 'actual_failed', 'predicted_failed', 'probability_failed']
PREDICTION_PROBABILITY_FLOAT_FORMAT = '%.11f'
FEATURE_NAME_MAP = {
    'X1': 'Current assets', 'X2': 'Cost of goods sold', 'X3': 'Depreciation and amortization',
    'X4': 'EBITDA', 'X5': 'Inventory', 'X6': 'Net income', 'X7': 'Total receivables',
    'X8': 'Market value', 'X9': 'Net sales', 'X10': 'Total assets', 'X11': 'Total long-term debt',
    'X12': 'EBIT', 'X13': 'Gross profit', 'X14': 'Total current liabilities',
    'X15': 'Retained earnings', 'X16': 'Total revenue', 'X17': 'Total liabilities',
    'X18': 'Total operating expenses',
}
FEATURE_DESCRIPTION_MAP = {
    'X1': 'Assets expected to be sold, converted into cash, or used within one year.',
    'X2': "Direct costs related to producing or selling the firm's goods and services.",
    'X3': 'Depreciation of tangible assets and amortization of intangible assets.',
    'X4': 'Earnings before interest, taxes, depreciation, and amortization.',
    'X5': 'Goods and raw materials held by the firm for production or sale.',
    'X6': 'Profit after expenses and costs have been deducted from revenue.',
    'X7': 'Money owed to the firm by customers for delivered goods or services.',
    'X8': 'Market capitalization or market value of the publicly traded company.',
    'X9': 'Gross sales minus returns, allowances, and discounts.',
    'X10': 'Total assets owned or controlled by the company.',
    'X11': 'Debt obligations due after more than one year.',
    'X12': 'Earnings before interest and taxes.',
    'X13': 'Profit after subtracting costs related to manufacturing and selling.',
    'X14': 'Short-term obligations due within one year.',
    'X15': 'Accumulated profits retained in the business after dividends and losses.',
    'X16': 'Total income from sales before subtracting expenses.',
    'X17': 'Total debts and obligations owed to outside parties.',
    'X18': 'Expenses incurred through normal business operations.',
}
MODEL_COLORS = {'Majority-class baseline': '#8c8c8c', 'Logistic Regression': '#4c78a8', 'Random Forest': '#59a14f', 'Gradient Boosting': '#f28e2b'}
OUTCOME_COLORS = {'detected': '#59a14f', 'missed': '#e15759', 'false_alarm': '#f28e2b'}
DIRECTION_COLORS = {'positive': '#c44e52', 'negative': '#4c78a8'}
METRIC_COLORS = {'Precision': '#4c78a8', 'Recall': '#e15759', 'F1': '#59a14f', 'ROC-AUC': '#4c78a8', 'PR-AUC': '#f28e2b', 'Failed F1': '#59a14f'}
BASELINE_COLOR = '#8c8c8c'
KEY_FEATURES_FOR_EDA = ['X8', 'X6', 'X11', 'X1', 'X17', 'X15']
KEY_MODELS_FOR_THRESHOLD_ANALYSIS = ['Logistic Regression', 'Random Forest', 'Gradient Boosting']

def apply_project_style():
    plt.rcParams.update({
        'font.family': 'DejaVu Sans', 'figure.facecolor': 'white', 'axes.facecolor': 'white',
        'axes.titlesize': 12, 'axes.labelsize': 10, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
        'legend.fontsize': 8.5, 'axes.spines.top': False, 'axes.spines.right': False,
        'grid.color': '#d9d9d9', 'grid.linewidth': 0.8, 'lines.linewidth': 2.0, 'lines.markersize': 5.5,
    })

def style_axis(ax, *, grid_axis='y', percent_y=False, integer_x=False):
    ax.grid(axis=grid_axis, linestyle='--', alpha=0.25)
    if percent_y:
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    if integer_x:
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))

def save_figure(fig, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def validate_required_columns(data):
    missing = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

def validate_target_labels(data):
    observed = set(data[RAW_TARGET_COLUMN].dropna().unique())
    unexpected = observed.difference({ALIVE_LABEL, FAILED_LABEL})
    if unexpected:
        raise ValueError(f'Unexpected target labels found: {sorted(unexpected)}')

def identify_feature_columns(data):
    excluded = {COMPANY_COLUMN, RAW_TARGET_COLUMN, YEAR_COLUMN}
    return [c for c in data.columns if c not in excluded and pd.api.types.is_numeric_dtype(data[c])]

def get_feature_columns(data, include_year=False):
    excluded = {COMPANY_COLUMN, TARGET_COLUMN}
    if not include_year:
        excluded.add(YEAR_COLUMN)
    return [c for c in data.columns if c not in excluded]

def split_features_target(data, include_year=False):
    feature_columns = get_feature_columns(data, include_year=include_year)
    return data[feature_columns].copy(), data[TARGET_COLUMN].copy()

def create_company_level_split(data, test_size=OUTER_TEST_SIZE, random_state=RANDOM_STATE):
    missing = {COMPANY_COLUMN, TARGET_COLUMN}.difference(data.columns)
    if missing:
        raise ValueError(f'Missing required split columns: {sorted(missing)}')
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(data, y=data[TARGET_COLUMN], groups=data[COMPANY_COLUMN]))
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

def check_no_company_overlap(left, right):
    return set(left[COMPANY_COLUMN].unique()).isdisjoint(set(right[COMPANY_COLUMN].unique()))

def create_split_summary(full_data, train_data, test_data):
    def summarize(name, data):
        return {'split': name, 'n_rows': int(len(data)), 'n_companies': int(data[COMPANY_COLUMN].nunique()), 'n_failed_rows': int(data[TARGET_COLUMN].sum()), 'n_alive_rows': int(len(data) - data[TARGET_COLUMN].sum()), 'failure_rate': float(data[TARGET_COLUMN].mean())}
    summary = pd.DataFrame([summarize('full', full_data), summarize('train', train_data), summarize('test', test_data)])
    summary['company_share'] = summary['n_companies'] / int(full_data[COMPANY_COLUMN].nunique())
    summary['row_share'] = summary['n_rows'] / int(len(full_data))
    return summary

def evaluate_binary_classifier(model_name, y_true, y_pred, probability_failed):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {'model': model_name, 'accuracy': accuracy_score(y_true, y_pred), 'balanced_accuracy': balanced_accuracy_score(y_true, y_pred), 'roc_auc': roc_auc_score(y_true, probability_failed), 'pr_auc': average_precision_score(y_true, probability_failed), 'precision_failed': precision_score(y_true, y_pred, zero_division=0), 'recall_failed': recall_score(y_true, y_pred, zero_division=0), 'f1_failed': f1_score(y_true, y_pred, zero_division=0), 'true_negative': int(tn), 'false_positive': int(fp), 'false_negative': int(fn), 'true_positive': int(tp)}

def create_classification_report_table(model_name, y_true, y_pred):
    report = classification_report(y_true, y_pred, labels=[0, 1], target_names=['alive', 'failed'], output_dict=True, zero_division=0)
    rows = []
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            rows.append({'model': model_name, 'class_or_average': label, **metrics})
    return pd.DataFrame(rows)

def create_prediction_table(data, model_name, y_pred, probability_failed):
    table = pd.DataFrame({'model': model_name, COMPANY_COLUMN: data[COMPANY_COLUMN].to_numpy(), YEAR_COLUMN: data[YEAR_COLUMN].to_numpy(), 'actual_failed': data[TARGET_COLUMN].to_numpy(), 'predicted_failed': y_pred, 'probability_failed': probability_failed})
    return table[PREDICTION_TABLE_COLUMNS]

def save_prediction_table(predictions, output_path):
    predictions[PREDICTION_TABLE_COLUMNS].to_csv(output_path, index=False, float_format=PREDICTION_PROBABILITY_FLOAT_FORMAT)

def get_probability_failed(model, x):
    return model.predict_proba(x)[:, 1]

def build_majority_class_baseline():
    return DummyClassifier(strategy='most_frequent')

def build_logistic_pipeline(penalty='l2', c_value=1.0, class_weight='balanced'):
    if penalty not in {'l1', 'l2'}:
        raise ValueError("Only 'l1' and 'l2' penalties are supported in this project.")
    l1_ratio = 1.0 if penalty == 'l1' else 0.0
    model = LogisticRegression(C=c_value, l1_ratio=l1_ratio, solver='saga', class_weight=class_weight, max_iter=5000, random_state=RANDOM_STATE)
    return Pipeline(steps=[('scaler', StandardScaler()), ('model', model)])

def build_interpretable_logit():
    return build_logistic_pipeline(penalty='l2', c_value=1.0)

def select_regularized_logit(x_train, y_train, x_valid, y_valid, penalty, c_grid):
    best_model, best_c, best_score = None, None, -1.0
    for c_value in c_grid:
        candidate = build_logistic_pipeline(penalty=penalty, c_value=c_value)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model, best_c, best_score = candidate, c_value, score
    if best_model is None:
        raise RuntimeError('No Logistic Regression model was selected.')
    return best_model, best_c, best_score

def build_decision_tree(max_depth=3, min_samples_leaf=100):
    return DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=min_samples_leaf, class_weight='balanced', random_state=RANDOM_STATE)

def build_random_forest(n_estimators=300, max_depth=5, min_samples_leaf=50, max_features='sqrt'):
    return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)

def build_gradient_boosting(n_estimators=150, learning_rate=0.05, max_depth=2, min_samples_leaf=100):
    return GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=RANDOM_STATE)

def select_decision_tree(x_train, y_train, x_valid, y_valid):
    grid = {'max_depth': [2, 3, 4, 5], 'min_samples_leaf': [50, 100, 200]}
    best_model, best_params, best_score = None, None, -1.0
    for max_depth, min_samples_leaf in product(grid['max_depth'], grid['min_samples_leaf']):
        candidate = build_decision_tree(max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {'max_depth': max_depth, 'min_samples_leaf': min_samples_leaf}
            best_score = score
    return best_model, best_params, best_score

def select_random_forest(x_train, y_train, x_valid, y_valid):
    grid = {'n_estimators': [300], 'max_depth': [4, 6, None], 'min_samples_leaf': [50, 100], 'max_features': ['sqrt']}
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, max_depth, min_samples_leaf, max_features in product(grid['n_estimators'], grid['max_depth'], grid['min_samples_leaf'], grid['max_features']):
        candidate = build_random_forest(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf, max_features=max_features)
        candidate.fit(x_train, y_train)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {'n_estimators': n_estimators, 'max_depth': max_depth, 'min_samples_leaf': min_samples_leaf, 'max_features': max_features}
            best_score = score
    return best_model, best_params, best_score

def select_gradient_boosting(x_train, y_train, x_valid, y_valid):
    grid = {'n_estimators': [100, 150], 'learning_rate': [0.03, 0.05], 'max_depth': [2, 3], 'min_samples_leaf': [100]}
    sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)
    best_model, best_params, best_score = None, None, -1.0
    for n_estimators, learning_rate, max_depth, min_samples_leaf in product(grid['n_estimators'], grid['learning_rate'], grid['max_depth'], grid['min_samples_leaf']):
        candidate = build_gradient_boosting(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, min_samples_leaf=min_samples_leaf)
        candidate.fit(x_train, y_train, sample_weight=sample_weight)
        score = average_precision_score(y_valid, candidate.predict_proba(x_valid)[:, 1])
        if score > best_score:
            best_model = candidate
            best_params = {'n_estimators': n_estimators, 'learning_rate': learning_rate, 'max_depth': max_depth, 'min_samples_leaf': min_samples_leaf}
            best_score = score
    return best_model, best_params, best_score

def parse_selected_parameters(value):
    if pd.isna(value) or value == '':
        return {}
    text = str(value)
    if text.startswith('{'):
        return ast.literal_eval(text)
    params = {}
    for piece in text.split(','):
        if '=' in piece:
            key, val = piece.strip().split('=', 1)
            if key == 'C':
                params['C'] = float(val)
            else:
                params[key] = val
    return params

def build_model_from_spec(model_name, selected_parameters):
    params = parse_selected_parameters(selected_parameters)
    if model_name == 'Majority-class baseline':
        return build_majority_class_baseline()
    if model_name == 'Logistic Regression':
        return build_interpretable_logit()
    if model_name == 'L1 Regularized Logistic Regression':
        return build_logistic_pipeline(penalty='l1', c_value=params['C'])
    if model_name == 'L2 Regularized Logistic Regression':
        return build_logistic_pipeline(penalty='l2', c_value=params['C'])
    if model_name == 'Decision Tree':
        return build_decision_tree(**params)
    if model_name == 'Random Forest':
        return build_random_forest(**params)
    if model_name == 'Gradient Boosting':
        return build_gradient_boosting(**params)
    raise ValueError(f'Unknown model: {model_name}')


## Internal Split

In [2]:

train_full = pd.read_csv(TRAIN_DATA_PATH)
model_train, validation = create_company_level_split(train_full, test_size=VALIDATION_SIZE, random_state=RANDOM_STATE)
assert check_no_company_overlap(model_train, validation), 'Internal split contains company overlap.'
assert set(model_train[COMPANY_COLUMN]).isdisjoint(set(validation[COMPANY_COLUMN])), 'Internal train and validation companies must be disjoint.'
assert len(model_train) + len(validation) == len(train_full), 'Internal split must preserve training rows.'
x_train, y_train = split_features_target(model_train)
x_valid, y_valid = split_features_target(validation)
pd.DataFrame({'sample': ['model_train', 'validation'], 'rows': [len(model_train), len(validation)], 'companies': [model_train[COMPANY_COLUMN].nunique(), validation[COMPANY_COLUMN].nunique()], 'failure_rate': [y_train.mean(), y_valid.mean()]})


,sample,rows,companies,failure_rate
0,model_train,50323,5740,0.063848
1,validation,12665,1436,0.065535


## Fit and Select Models

In [3]:

models = {}
model_specs = []
majority_model = build_majority_class_baseline(); majority_model.fit(x_train, y_train)
models['Majority-class baseline'] = majority_model
model_specs.append({'model': 'Majority-class baseline', 'model_type': 'DummyClassifier', 'model_family': 'baseline', 'selected_parameters': '', 'selection_metric': '', 'validation_pr_auc_during_selection': ''})
logit = build_interpretable_logit(); logit.fit(x_train, y_train)
models['Logistic Regression'] = logit
model_specs.append({'model': 'Logistic Regression', 'model_type': 'LogisticRegression', 'model_family': 'logistic_regression', 'selected_parameters': 'penalty=l2, C=1.0', 'selection_metric': 'fixed benchmark', 'validation_pr_auc_during_selection': ''})
l1, l1_c, l1_score = select_regularized_logit(x_train, y_train, x_valid, y_valid, 'l1', LOGISTIC_C_GRID)
models['L1 Regularized Logistic Regression'] = l1
model_specs.append({'model': 'L1 Regularized Logistic Regression', 'model_type': 'LogisticRegression', 'model_family': 'logistic_regression', 'selected_parameters': f'penalty=l1, C={l1_c}', 'selection_metric': 'validation PR-AUC', 'validation_pr_auc_during_selection': l1_score})
l2, l2_c, l2_score = select_regularized_logit(x_train, y_train, x_valid, y_valid, 'l2', LOGISTIC_C_GRID)
models['L2 Regularized Logistic Regression'] = l2
model_specs.append({'model': 'L2 Regularized Logistic Regression', 'model_type': 'LogisticRegression', 'model_family': 'logistic_regression', 'selected_parameters': f'penalty=l2, C={l2_c}', 'selection_metric': 'validation PR-AUC', 'validation_pr_auc_during_selection': l2_score})
dt, dt_params, dt_score = select_decision_tree(x_train, y_train, x_valid, y_valid)
models['Decision Tree'] = dt
model_specs.append({'model': 'Decision Tree', 'model_type': 'DecisionTreeClassifier', 'model_family': 'tree_based', 'selected_parameters': str(dt_params), 'selection_metric': 'validation PR-AUC', 'validation_pr_auc_during_selection': dt_score})
rf, rf_params, rf_score = select_random_forest(x_train, y_train, x_valid, y_valid)
models['Random Forest'] = rf
model_specs.append({'model': 'Random Forest', 'model_type': 'RandomForestClassifier', 'model_family': 'tree_based', 'selected_parameters': str(rf_params), 'selection_metric': 'validation PR-AUC', 'validation_pr_auc_during_selection': rf_score})
gb, gb_params, gb_score = select_gradient_boosting(x_train, y_train, x_valid, y_valid)
models['Gradient Boosting'] = gb
model_specs.append({'model': 'Gradient Boosting', 'model_type': 'GradientBoostingClassifier', 'model_family': 'tree_based', 'selected_parameters': str(gb_params), 'selection_metric': 'validation PR-AUC', 'validation_pr_auc_during_selection': gb_score})
assert list(models.keys()) == EXPECTED_MODEL_ORDER, 'Unexpected model order or names.'
assert l1_c in LOGISTIC_C_GRID and l2_c in LOGISTIC_C_GRID, 'Selected logistic C must come from declared grid.'
pd.DataFrame(model_specs)


,model,model_type,model_family,selected_parameters,selection_metric,validation_pr_auc_during_selection
0,Majority-class baseline,DummyClassifier,baseline,,,
1,Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=1.0",fixed benchmark,
2,L1 Regularized Logistic Regression,LogisticRegression,logistic_regression,"penalty=l1, C=0.1",validation PR-AUC,0.148295
3,L2 Regularized Logistic Regression,LogisticRegression,logistic_regression,"penalty=l2, C=0.1",validation PR-AUC,0.14873
4,Decision Tree,DecisionTreeClassifier,tree_based,"{'max_depth': 5, 'min_samples_leaf': 50}",validation PR-AUC,0.146024
5,Random Forest,RandomForestClassifier,tree_based,"{'n_estimators': 300, 'max_depth': None, 'min_...",validation PR-AUC,0.154216
6,Gradient Boosting,GradientBoostingClassifier,tree_based,"{'n_estimators': 150, 'learning_rate': 0.05, '...",validation PR-AUC,0.174674


## Validation Evaluation

In [4]:

metric_rows, report_tables, prediction_tables = [], [], []
for model_name, model in models.items():
    y_pred = model.predict(x_valid)
    probability_failed = get_probability_failed(model, x_valid)
    assert len(probability_failed) == len(validation), f'{model_name} probability length mismatch.'
    assert pd.Series(probability_failed).between(0, 1).all(), f'{model_name} probabilities outside [0, 1].'
    metric_rows.append(evaluate_binary_classifier(model_name, y_valid, y_pred, probability_failed))
    report_tables.append(create_classification_report_table(model_name, y_valid, y_pred))
    prediction_tables.append(create_prediction_table(validation, model_name, y_pred, probability_failed))
validation_model_comparison = pd.DataFrame(metric_rows)
validation_classification_reports = pd.concat(report_tables, ignore_index=True)
model_specification = pd.DataFrame(model_specs)
validation_predictions = pd.concat(prediction_tables, ignore_index=True)
assert list(validation_model_comparison['model']) == EXPECTED_MODEL_ORDER, 'Validation model order changed.'
assert validation_predictions.groupby('model').size().eq(len(validation)).all(), 'Each model must produce one validation prediction per row.'
for _, row in validation_model_comparison.iterrows():
    assert row[['true_negative', 'false_positive', 'false_negative', 'true_positive']].sum() == len(validation), f'Confusion counts inconsistent for {row.model}.'
validation_model_comparison.to_csv(TABLES_DIR / 'validation_model_comparison.csv', index=False)
validation_classification_reports.to_csv(TABLES_DIR / 'validation_classification_reports.csv', index=False)
model_specification.to_csv(TABLES_DIR / 'model_specification.csv', index=False)
save_prediction_table(validation_predictions, TABLES_DIR / 'validation_predictions.csv')
validation_model_comparison


,model,accuracy,balanced_accuracy,roc_auc,pr_auc,precision_failed,recall_failed,f1_failed,true_negative,false_positive,false_negative,true_positive
0,Majority-class baseline,0.934465,0.500000,0.500000,0.065535,0.000000,0.000000,0.000000,11835,0,830,0
1,Logistic Regression,0.366996,0.589600,0.671009,0.147629,0.081713,0.845783,0.149029,3946,7889,128,702
2,L1 Regularized Logistic Regression,0.367075,0.592444,0.671667,0.148295,0.082209,0.851807,0.149947,3942,7893,123,707
3,L2 Regularized Logistic Regression,0.366285,0.592021,0.672404,0.148730,0.082114,0.851807,0.149788,3932,7903,123,707
4,Decision Tree,0.604737,0.587410,0.643907,0.146024,0.092028,0.567470,0.158373,7188,4647,359,471
5,Random Forest,0.816502,0.608852,0.702748,0.154216,0.145636,0.369880,0.208986,10034,1801,523,307
6,Gradient Boosting,0.677852,0.627652,0.691807,0.174674,0.112726,0.569880,0.188221,8112,3723,357,473
